# 003 Börde Non-DL Spatial Allocation Method Comparison

Unified evaluation of 35 non-deep-learning spatial load allocation methods, covering:

**Baselines (3)**: uniform_average, gemeinde_equal, gemeinde_area

**Euclidean distance + demand weighting (4)**: voronoi, civd, voronoi_gpm, civd_gpm

**WorldCover correction (2)**: voronoi_wc_gpm, civd_wc_gpm

**NTL nighttime-lights correction (5)**: voronoi_ntl, civd_ntl, voronoi_ntl_gpm, civd_ntl_gpm, voronoi_wc_ntl_gpm

**Substation-proximity correction -- Prox only (8)**:
- gamma=1: voronoi_prox1, civd_prox1, voronoi_prox1_gpm, civd_prox1_gpm
- gamma=2: voronoi_prox2, civd_prox2, voronoi_prox2_gpm, civd_prox2_gpm

**Substation-proximity correction -- NTL+Prox (8)**:
- gamma=1: voronoi_prox1_ntl, civd_prox1_ntl, voronoi_prox1_ntl_gpm, civd_prox1_ntl_gpm
- gamma=2: voronoi_prox2_ntl, civd_prox2_ntl, voronoi_prox2_ntl_gpm, civd_prox2_ntl_gpm

**Network distance (4, conditional)**: voronoi_ND, civd_ND, voronoi_gpm_ND, civd_gpm_ND

**Oracle (1)**: gemeinde_average

> Note: the German Börde case is a single region, with 34 Gemeinden as source nodes and
> 13 substations as targets. Unlike the UK case, there is no ITL2/ITL3 hierarchy here;
> the Gemeinde level is used instead.

In [ ]:
import sys
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error, mean_absolute_error

PROJECT_ROOT = Path('../../').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from SpatialAllocation.Allocator import allocator_registry
from SpatialAllocation.Allocator.clustering.do_clustering import do_clustering
from SpatialAllocation.Weighter import weighter_registry
from SpatialAllocation.FeatureExtractor.correctors.proximity_corrector import ProximityCorrector

try:
    from SpatialAllocation.utils.NetworkDistance import load_distance_results
    _nd_import_ok = True
except ImportError:
    _nd_import_ok = False

warnings.filterwarnings('ignore', category=FutureWarning)

# ─── Path constants ───
DATA_DIR = Path('./results/intermediate')
ASSEMBLED_DIR = DATA_DIR / 'features' / 'assembled'
EXTRACTED_DIR = DATA_DIR / 'features' / 'extracted'
ND_DATA_DIR = DATA_DIR / 'features' / 'network_distance'
OUTPUT_DIR = Path('./results/static_allocation')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ─── Column name constants ───
RELATION_COL = 'Name'           # Gemeinde name (shared by grid + source_regions)
DEMAND_COL = 'p_mw'             # Actual substation load
DERIVED_DEMAND_COL = 'Demand (MVA)'
TARGET_CRS = 'EPSG:25832'
DIST_CLAMP_KM = 0.01
GAMMA_VALUES = [1.0, 2.0]
RCI_THRESHOLD = 0.5

LANDUSE_PERCENT_MAP = {
    'lu_residential_prop': 'residential_percent',
    'lu_commercial_prop': 'commercial_percent',
    'lu_industrial_prop': 'industrial_percent',
    'lu_agricultural_prop': 'agricultural_percent',
    'lu_others_prop': 'others_percent',
}
LU_COLS = list(LANDUSE_PERCENT_MAP.keys())
PCT_COLS = list(LANDUSE_PERCENT_MAP.values())

print('Imports complete')

In [ ]:
# ─── Load data ───
region_gdf = gpd.read_file(str(DATA_DIR / 'source_regions.gpkg'))
subs_gdf = gpd.read_file(str(DATA_DIR / 'substations.gpkg'))

# Derive Gemeinde-level load
gemeinde_demand = subs_gdf.groupby('Gemeinde')[DEMAND_COL].sum()
region_gdf[DERIVED_DEMAND_COL] = region_gdf[RELATION_COL].map(gemeinde_demand).fillna(0.0)

n_with = (region_gdf[DERIVED_DEMAND_COL] > 0).sum()
print(f'Gemeinden: {len(region_gdf)}, with load: {n_with}')
print(f'Substations: {len(subs_gdf)}, total load: {subs_gdf[DEMAND_COL].sum():.2f} MW')
print(f'Derived total Gemeinde load: {region_gdf[DERIVED_DEMAND_COL].sum():.2f} MW')

# Load grid points
with open(ASSEMBLED_DIR / 'boerde_grid_points.pickle', 'rb') as f:
    grid_gdf, step_size_m = pickle.load(f)
print(f'Grid points: {len(grid_gdf)}, step={step_size_m}m')

# NTL
ntl_npz = np.load(EXTRACTED_DIR / 'boerde_ntl.npz', allow_pickle=True)
ntl_values = ntl_npz['data'][:, 0]
assert len(ntl_values) == len(grid_gdf)

# ND
nd_dir = ND_DATA_DIR / 'boerde'
nd_available = nd_dir.exists() and _nd_import_ok
print(f'NTL: {len(ntl_values)} points, ND available: {nd_available}')

In [ ]:
# ─── Helper functions ───

def compute_demand(grid_gdf, region_sub, weighter_result, demand_col='demand'):
    """Use weighter weights + regional percentages to derive grid demand (normalized per Gemeinde)."""
    W = weighter_result.weights
    gdf = grid_gdf.copy()
    gdf[demand_col] = 0.0
    region_info = region_sub.set_index(RELATION_COL)
    for name, group in gdf.groupby(RELATION_COL):
        if name not in region_info.index:
            continue
        total_demand = region_info.loc[name, DERIVED_DEMAND_COL]
        idx = group.index
        if W.ndim == 2:
            pcts = np.array([region_info.loc[name, c] for c in PCT_COLS])
            score = W[idx] @ pcts
        else:
            score = W[idx]
        score_sum = score.sum()
        if score_sum > 0:
            gdf.loc[idx, demand_col] = total_demand * score / score_sum
        else:
            gdf.loc[idx, demand_col] = total_demand / len(group)
    return gdf


def ntl_correct(grid_gdf, region_sub, base_col, ntl_vals, out_col):
    """NTL post-correction: base × ntl_factor, normalized per Gemeinde."""
    rci_sum = (grid_gdf['lu_residential_prop'].values
               + grid_gdf['lu_commercial_prop'].values
               + grid_gdf['lu_industrial_prop'].values)
    rci_mask = rci_sum > RCI_THRESHOLD
    grid_gdf[out_col] = 0.0
    region_info = region_sub.set_index(RELATION_COL)
    for name, group in grid_gdf.groupby(RELATION_COL):
        if name not in region_info.index:
            continue
        total = region_info.loc[name, DERIVED_DEMAND_COL]
        idx = group.index
        ntl_g = ntl_vals[idx]
        rci_g = rci_mask[idx]
        rci_nz = ntl_g[rci_g & (ntl_g > 0)]
        eps = np.percentile(rci_nz, 5) if len(rci_nz) > 0 else (
            np.percentile(ntl_g[ntl_g > 0], 5) if (ntl_g > 0).any() else 0.1)
        rci_ntl = ntl_g[rci_g]
        med = np.median(rci_ntl) if len(rci_ntl) > 0 else np.median(ntl_g)
        if med <= 0:
            med = eps
        factor = np.log(1 + ntl_g + eps) / np.log(1 + med)
        base = grid_gdf.loc[idx, base_col].values
        raw = base * factor
        s = raw.sum()
        if s > 0:
            grid_gdf.loc[idx, out_col] = total * raw / s
        else:
            grid_gdf.loc[idx, out_col] = total / len(group)


def prox_correct(grid_gdf, region_sub, base_col, prox_scores, out_col):
    """Proximity post-correction: base × prox_factor, normalized per Gemeinde."""
    rci_sum = (grid_gdf['lu_residential_prop'].values
               + grid_gdf['lu_commercial_prop'].values
               + grid_gdf['lu_industrial_prop'].values)
    rci_mask = rci_sum > RCI_THRESHOLD
    grid_gdf[out_col] = 0.0
    region_info = region_sub.set_index(RELATION_COL)
    for name, group in grid_gdf.groupby(RELATION_COL):
        if name not in region_info.index:
            continue
        total = region_info.loc[name, DERIVED_DEMAND_COL]
        idx = group.index
        pg = prox_scores[idx]
        rci_g = rci_mask[idx]
        rci_p = pg[rci_g]
        med = np.median(rci_p) if len(rci_p) > 0 else np.median(pg)
        if med <= 0:
            med = 1e-6
        factor = np.log(1 + pg) / np.log(1 + med)
        base = grid_gdf.loc[idx, base_col].values
        raw = base * factor
        s = raw.sum()
        if s > 0:
            grid_gdf.loc[idx, out_col] = total * raw / s
        else:
            grid_gdf.loc[idx, out_col] = total / len(group)


def wc_correct(grid_gdf, region_sub, base_col, wc_others, out_col):
    """WorldCover correction: base × (1 - wc_others), normalized per Gemeinde."""
    grid_gdf[out_col] = grid_gdf[base_col] * (1 - wc_others)
    region_info = region_sub.set_index(RELATION_COL)
    for name, group in grid_gdf.groupby(RELATION_COL):
        if name not in region_info.index:
            continue
        total = region_info.loc[name, DERIVED_DEMAND_COL]
        idx = group.index
        s = grid_gdf.loc[idx, out_col].sum()
        if s > 0:
            grid_gdf.loc[idx, out_col] *= total / s
        else:
            grid_gdf.loc[idx, out_col] = total / len(group)


def aggregate_to_substations(grid_gdf, subs_gdf, assignment, demand_col):
    result = subs_gdf.copy()
    result['allocated_demand'] = 0.0
    demands = grid_gdf[demand_col].values
    for t_idx in range(len(subs_gdf)):
        mask = assignment == t_idx
        result.loc[t_idx, 'allocated_demand'] = demands[mask].sum()
    return result


def aggregate_clustered(grid_gdf, subs_gdf, cluster_gdf, assignment, demand_col):
    result = subs_gdf.copy()
    result['allocated_demand'] = 0.0
    demands = grid_gdf[demand_col].values
    cluster_demands = {}
    for label in np.unique(assignment):
        cluster_demands[label] = demands[assignment == label].sum()
    for label, total_d in cluster_demands.items():
        members = cluster_gdf[cluster_gdf['cluster_label'] == label].index
        n = len(members)
        if n > 0:
            for idx in members:
                if idx < len(result):
                    result.loc[idx, 'allocated_demand'] += total_d / n
    return result


def evaluate_allocation(subs_result, actual_col=DEMAND_COL, alloc_col='allocated_demand'):
    actual = subs_result[actual_col].values
    allocated = subs_result[alloc_col].values
    corr, _ = pearsonr(actual, allocated)
    rmse = np.sqrt(mean_squared_error(actual, allocated))
    mae = mean_absolute_error(actual, allocated)
    return {'corr': corr, 'rmse': rmse, 'mae': mae}


def reconstruct_full_nd_matrix(nd_matrix, nd_target_indices, n_targets):
    n_agents, k = nd_matrix.shape
    full = np.full((n_agents, n_targets), np.inf, dtype=np.float64)
    rows = np.repeat(np.arange(n_agents), k)
    cols = nd_target_indices.ravel()
    vals = nd_matrix.ravel()
    valid = cols >= 0
    full[rows[valid], cols[valid]] = vals[valid]
    return full

print('Helper functions defined')

In [ ]:
METHOD_ORDER = [
    # Baselines
    'uniform_average', 'gemeinde_equal', 'gemeinde_area',
    # Euclidean + uncorrected
    'voronoi', 'civd', 'voronoi_gpm', 'civd_gpm',
    # WC
    'voronoi_wc_gpm', 'civd_wc_gpm',
    # NTL
    'voronoi_ntl', 'civd_ntl',
    'voronoi_ntl_gpm', 'civd_ntl_gpm',
    'voronoi_wc_ntl_gpm',
    # Proximity only γ=1
    'voronoi_prox1', 'civd_prox1',
    'voronoi_prox1_gpm', 'civd_prox1_gpm',
    # Proximity only γ=2
    'voronoi_prox2', 'civd_prox2',
    'voronoi_prox2_gpm', 'civd_prox2_gpm',
    # NTL+Proximity γ=1
    'voronoi_prox1_ntl', 'civd_prox1_ntl',
    'voronoi_prox1_ntl_gpm', 'civd_prox1_ntl_gpm',
    # NTL+Proximity γ=2
    'voronoi_prox2_ntl', 'civd_prox2_ntl',
    'voronoi_prox2_ntl_gpm', 'civd_prox2_ntl_gpm',
    # ND (conditional)
    'voronoi_ND', 'civd_ND',
    'voronoi_gpm_ND', 'civd_gpm_ND',
    # Oracle
    'gemeinde_average',
]

In [ ]:
# ─── Main computation: single region boerde ───

results = {}
total_demand = region_gdf[DERIVED_DEMAND_COL].sum()

# ── 1. Base demand ──
uniform = weighter_registry.create('uniform', config={})
uniform_res = uniform.compute(grid_gdf, target_gdf=subs_gdf)
grid_gdf = compute_demand(grid_gdf, region_gdf, uniform_res, demand_col='average_demand')

gpm = weighter_registry.create('gpm', config={
    'mode': 'categorical', 'proportion_columns': LU_COLS,
})
gpm_res = gpm.compute(grid_gdf, target_gdf=subs_gdf)
grid_gdf = compute_demand(grid_gdf, region_gdf, gpm_res, demand_col='landuse_demand')

# ── 2. WC correction ──
wc_correct(grid_gdf, region_gdf, 'landuse_demand',
           grid_gdf['wc_others_ratio'].values, 'wc_landuse_demand')

# ── 3. NTL correction ──
ntl_correct(grid_gdf, region_gdf, 'average_demand', ntl_values, 'ntl_average_demand')
ntl_correct(grid_gdf, region_gdf, 'landuse_demand', ntl_values, 'ntl_landuse_demand')
ntl_correct(grid_gdf, region_gdf, 'wc_landuse_demand', ntl_values, 'wc_ntl_landuse_demand')

# ── 4. Proximity correction ──
for gamma in GAMMA_VALUES:
    g_tag = f'prox{int(gamma)}'
    prox_scores = ProximityCorrector.compute_scores(
        grid_gdf, subs_gdf, gamma=gamma,
        target_crs=TARGET_CRS, clamp_km=DIST_CLAMP_KM)
    prox_correct(grid_gdf, region_gdf, 'ntl_average_demand',
                 prox_scores, f'{g_tag}_ntl_average_demand')
    prox_correct(grid_gdf, region_gdf, 'ntl_landuse_demand',
                 prox_scores, f'{g_tag}_ntl_landuse_demand')
    # Prox-only (no NTL) ablation
    prox_correct(grid_gdf, region_gdf, 'average_demand',
                 prox_scores, f'{g_tag}_average_demand')
    prox_correct(grid_gdf, region_gdf, 'landuse_demand',
                 prox_scores, f'{g_tag}_landuse_demand')

# ── 5. Demand conservation check ──
check_cols = [
    'average_demand', 'landuse_demand', 'wc_landuse_demand',
    'ntl_average_demand', 'ntl_landuse_demand', 'wc_ntl_landuse_demand',
    'prox1_ntl_average_demand', 'prox1_ntl_landuse_demand',
    'prox2_ntl_average_demand', 'prox2_ntl_landuse_demand',
    'prox1_average_demand', 'prox1_landuse_demand',
    'prox2_average_demand', 'prox2_landuse_demand',
]
for col in check_cols:
    err = abs(grid_gdf[col].sum() - total_demand)
    assert err < 1.0, f'{col}: conservation error {err:.4f}'
print(f'Demand conservation check passed (total {total_demand:.2f} MW)')

# ── 6. Voronoi allocation ──
alloc = allocator_registry.create('voronoi')
voronoi_res = alloc.allocate(grid_gdf, subs_gdf)

voronoi_methods = [
    ('voronoi', 'average_demand'),
    ('voronoi_gpm', 'landuse_demand'),
    ('voronoi_wc_gpm', 'wc_landuse_demand'),
    ('voronoi_ntl', 'ntl_average_demand'),
    ('voronoi_ntl_gpm', 'ntl_landuse_demand'),
    ('voronoi_wc_ntl_gpm', 'wc_ntl_landuse_demand'),
    ('voronoi_prox1', 'prox1_average_demand'),
    ('voronoi_prox1_gpm', 'prox1_landuse_demand'),
    ('voronoi_prox2', 'prox2_average_demand'),
    ('voronoi_prox2_gpm', 'prox2_landuse_demand'),
    ('voronoi_prox1_ntl', 'prox1_ntl_average_demand'),
    ('voronoi_prox1_ntl_gpm', 'prox1_ntl_landuse_demand'),
    ('voronoi_prox2_ntl', 'prox2_ntl_average_demand'),
    ('voronoi_prox2_ntl_gpm', 'prox2_ntl_landuse_demand'),
]
for method, dcol in voronoi_methods:
    results[method] = aggregate_to_substations(
        grid_gdf, subs_gdf, voronoi_res.assignment, dcol)

# ── 7. CIVD allocation ──
coords = np.column_stack([subs_gdf.geometry.x.values, subs_gdf.geometry.y.values])
cluster_gdf_c, centroid_gdf_c = do_clustering(coords, method='hdbscan', min_cluster_size=2)

target_civd = subs_gdf.copy()
target_civd['cluster_label'] = cluster_gdf_c['cluster_label']

civd_config = {
    'solver': 'scip', 'method': 'civd',
    'cluster_label_column': 'cluster_label', 'n_jobs': -1,
}
alloc_civd = allocator_registry.create('civd', config=civd_config)
civd_res = alloc_civd.allocate(grid_gdf, target_civd)
civd_assignment = civd_res.assignment

# Cache CIVD
civd_cache = OUTPUT_DIR / 'boerde_civd_cache.pickle'
with open(civd_cache, 'wb') as f:
    pickle.dump({
        'assignment': civd_assignment,
        'n_grid': len(grid_gdf),
        'n_target': len(target_civd),
        'config': civd_config,
    }, f)

civd_methods = [
    ('civd', 'average_demand'),
    ('civd_gpm', 'landuse_demand'),
    ('civd_wc_gpm', 'wc_landuse_demand'),
    ('civd_ntl', 'ntl_average_demand'),
    ('civd_ntl_gpm', 'ntl_landuse_demand'),
    ('civd_prox1', 'prox1_average_demand'),
    ('civd_prox1_gpm', 'prox1_landuse_demand'),
    ('civd_prox2', 'prox2_average_demand'),
    ('civd_prox2_gpm', 'prox2_landuse_demand'),
    ('civd_prox1_ntl', 'prox1_ntl_average_demand'),
    ('civd_prox1_ntl_gpm', 'prox1_ntl_landuse_demand'),
    ('civd_prox2_ntl', 'prox2_ntl_average_demand'),
    ('civd_prox2_ntl_gpm', 'prox2_ntl_landuse_demand'),
]
for method, dcol in civd_methods:
    results[method] = aggregate_clustered(
        grid_gdf, subs_gdf, cluster_gdf_c, civd_assignment, dcol)

# ── 8. ND allocation (conditional) ──
if nd_available:
    nd_matrix, nd_target_idx, _, nd_target_map = load_distance_results(str(nd_dir))
    full_nd = reconstruct_full_nd_matrix(nd_matrix, nd_target_idx, len(subs_gdf))
    nd_assignment = full_nd.argmin(axis=1)
    results['voronoi_ND'] = aggregate_to_substations(
        grid_gdf, subs_gdf, nd_assignment, 'average_demand')
    results['voronoi_gpm_ND'] = aggregate_to_substations(
        grid_gdf, subs_gdf, nd_assignment, 'landuse_demand')

    alloc_nd = allocator_registry.create('civd', config=civd_config)
    nd_civd_res = alloc_nd.allocate(grid_gdf, target_civd, distance_matrix=full_nd)
    results['civd_ND'] = aggregate_clustered(
        grid_gdf, subs_gdf, cluster_gdf_c, nd_civd_res.assignment, 'average_demand')
    results['civd_gpm_ND'] = aggregate_clustered(
        grid_gdf, subs_gdf, cluster_gdf_c, nd_civd_res.assignment, 'landuse_demand')

# ── 9. Baseline methods ──
# uniform_average: total demand / number of substations
uniform_avg = subs_gdf.copy()
uniform_avg['allocated_demand'] = total_demand / len(subs_gdf)
results['uniform_average'] = uniform_avg

# gemeinde_equal: total demand split evenly across Gemeinden with substations, then evenly within each Gemeinde
gemeinden_with_subs = subs_gdf['Gemeinde'].unique()
n_gem_with = len(gemeinden_with_subs)
gem_equal = subs_gdf.copy()
gem_equal['allocated_demand'] = 0.0
for gem in gemeinden_with_subs:
    gem_subs = gem_equal[gem_equal['Gemeinde'] == gem].index
    gem_equal.loc[gem_subs, 'allocated_demand'] = (total_demand / n_gem_with) / len(gem_subs)
results['gemeinde_equal'] = gem_equal

# gemeinde_area: allocated by area share among Gemeinden with substations
gem_area = subs_gdf.copy()
gem_area['allocated_demand'] = 0.0
regions_with_subs = region_gdf[region_gdf[RELATION_COL].isin(gemeinden_with_subs)]
total_area = regions_with_subs['area_m2'].sum()
for _, reg in regions_with_subs.iterrows():
    gem_name = reg[RELATION_COL]
    area_ratio = reg['area_m2'] / total_area
    gem_demand = total_demand * area_ratio
    gem_subs = gem_area[gem_area['Gemeinde'] == gem_name].index
    if len(gem_subs) > 0:
        gem_area.loc[gem_subs, 'allocated_demand'] = gem_demand / len(gem_subs)
results['gemeinde_area'] = gem_area

# gemeinde_average (oracle): known Gemeinde load, split evenly within each Gemeinde
gem_avg = subs_gdf.copy()
gem_avg['allocated_demand'] = 0.0
for gem, group in gem_avg.groupby('Gemeinde'):
    region_row = region_gdf[region_gdf[RELATION_COL] == gem]
    if not region_row.empty:
        gem_avg.loc[group.index, 'allocated_demand'] = (
            region_row[DERIVED_DEMAND_COL].iloc[0] / len(group))
results['gemeinde_average'] = gem_avg

# ── 10. Evaluation ──
metrics = {}
for method_name, subs_result in results.items():
    metrics[method_name] = evaluate_allocation(subs_result)

print(f'Computation complete: {len(metrics)} methods')

In [ ]:
# ─── Summary table ───

metric_names = ['rmse', 'mae', 'corr']
summary = {}
for method_name, m in metrics.items():
    summary[method_name] = {k: round(m[k], 4) for k in metric_names}

summary_df = pd.DataFrame(summary).T
available = [m for m in METHOD_ORDER if m in summary_df.index]
summary_df = summary_df.loc[available]
summary_df.index.name = 'method'

display(summary_df)

# Save
summary_df.to_csv(OUTPUT_DIR / 'boerde_static_metrics.csv')
print(f'\nSaved to {OUTPUT_DIR / "boerde_static_metrics.csv"}')
print(f'Total {len(summary_df)} methods x {len(metric_names)} metrics')